In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

In [3]:
import logging
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

In [4]:
!pip install transformers torch --quiet

In [5]:

import warnings
warnings.filterwarnings('ignore')
import torch
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

Model loaded successfully.
Device map: Single device


In [6]:
def generate_reply(chat_history, user_message, max_new_tokens=256):
    # Append the new user message to the history
    chat_history.append({"role": "user", "content": user_message})
    # Apply Qwen chat template to convert messages into a single text prompt
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,  # model expects a generation prompt
    )
    # Tokenize the prompt
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)
    # Generate the model output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )


    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply

In [7]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]
    while True:
        user_input = input("User: ").strip()
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue
        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."
        print("Chatbot:", assistant_reply)
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: hello
Chatbot: Hello! How can I help you today?
User: what is ai
Chatbot: Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think, learn, and perform tasks without explicit instructions. It involves the use of algorithms, statistical models, and machine learning techniques to enable computers to understand, analyze, and respond to human or natural language inputs. AI systems can be used for a wide range of applications, from voice recognition and image processing to autonomous vehicles and expert systems.
User: who created python
Chatbot: Python was created by Guido van Rossum. He began working on Python in the late 1980s as a successor to the ABC programming language. The first public release of Python was in 1991. Guido van Rossum is also known for creating the Debian operating system and the Python web framework Django.
User: thank you 
Chatbot: You're